# Columns:
- time, device, active_power, irradiance, clear_sky_irradiance, ambient_temp,
- module_temp, inverter_temp, dc_voltage, dc_current, ac_voltage, ac_current,
- performance_ratio, cloud_cover, sun_elevation, sun_azimuth, day_length_hours,
- fault_soiling, fault_inverter_overheat, fault_tracker_stuck, fault_dc_string_outage,
- is_faulted, fault_labels, downtime_start_time, fault_severity

# Devices:
- PV000: healthy and clean
- PV001: summer soiling buildup
- PV002: summer inverter overheating with intermittent trips
- PV003: tracker actuator stuck with maintenance outage
- PV004: autumn string outage with short maintenance downtime
- PV005: early year soiling buildup, cleaned later
- PV006: tracker stuck mid year, repaired later
- PV007: dc string outage, fixed after replacement
- PV008: late summer inverter overheating, fixed after fan service
- PV009: autumn soiling buildup, cleaned at year end
- PV010: two drifts in sequence, soiling then tracker stuck, both fixed

# INFO:
- 10 minute intervals
- active_power follows the supplied irradiance to power curve
- sunrise, sunset, solar elevation, and day length vary through the year
- cloud cover, storms, and hot summer afternoons affect irradiance and temperatures
- power is zero at night and never exceeds the rated clipping plateau
- is_faulted = 1 whenever an injected fault is active, including ramp periods and outages


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
random_seed = 42

In [3]:
POWER_CURVE = [
    (0, 4.491939776),
    (10, 50.3),
    (20, 92.425),
    (30, 136.525),
    (40, 183.075),
    (50, 227.775),
    (60, 270.95),
    (70, 316.84),
    (80, 361.025),
    (90, 407.675),
    (100, 451.425),
    (110, 494.175),
    (120, 536.575),
    (130, 575.675),
    (140, 616.6),
    (150, 658.925),
    (160, 700.375),
    (170, 741.425),
    (180, 782.66),
    (190, 825.625),
    (200, 869.505),
    (210, 910.325),
    (220, 953.05),
    (230, 997.69),
    (240, 1040.575),
    (250, 1080.95),
    (260, 1122.225),
    (270, 1164.075),
    (280, 1205.5),
    (290, 1247.45),
    (300, 1288.25),
    (310, 1329.715),
    (320, 1372.76),
    (330, 1417.265),
    (340, 1459.125),
    (350, 1500.935),
    (360, 1543.2),
    (370, 1582.975),
    (380, 1621.5),
    (390, 1663),
    (400, 1704.075),
    (410, 1744.75),
    (420, 1783.55),
    (430, 1824.76),
    (440, 1867.475),
    (450, 1908.075),
    (460, 1950.25),
    (470, 1991.78),
    (480, 2031.95),
    (490, 2073),
    (500, 2113.375),
    (510, 2155.16),
    (520, 2197.5),
    (530, 2237),
    (540, 2276.815),
    (550, 2318),
    (560, 2359.45),
    (570, 2402.225),
    (580, 2445.75),
    (590, 2488.55),
    (600, 2529),
    (610, 2572),
    (620, 2612.725),
    (630, 2656.25),
    (640, 2698.2),
    (650, 2742.55),
    (660, 2784.55),
    (670, 2825.33),
    (680, 2867.325),
    (690, 2905.89),
    (700, 2945.05),
    (710, 2984.575),
    (720, 3023),
    (730, 3062.03),
    (740, 3101.84),
    (750, 3139.8),
    (760, 3177.5),
    (770, 3214.875),
    (780, 3250.975),
    (790, 3286.54675),
    (800, 3322.25),
    (810, 3358.125),
    (820, 3394.05),
    (830, 3431.825),
    (840, 3471.65),
    (850, 3512.475),
    (860, 3551.125),
    (870, 3591.155),
    (880, 3629.935),
    (890, 3669.375),
    (900, 3707.5),
    (910, 3745.9),
    (920, 3786.625),
    (930, 3826.53),
    (940, 3867.3),
    (950, 3904.8),
    (960, 3942.05),
    (970, 3979),
    (980, 4015),
    (990, 4052.4),
    (1000, 4087),
    (1010, 4124.5),
    (1020, 4164.85),
    (1030, 4200),
    (1040, 4234),
    (1050, 4265.78),
    (1060, 4294.325),
    (1070, 4319.375),
    (1080, 4343),
    (1090, 4363.125),
    (1100, 4378.5),
    (1110, 4390),
    (1120, 4395.5),
    (1130, 4400),
    (1140, 4400),
    (1150, 4400),
    (1160, 4400),
    (1170, 4400),
    (1180, 4400),
    (1200, 4400),
    (1210, 4400),
    (1220, 4400),
    (1230, 4400),
    (1240, 4400),
    (1250, 4400),
]

IRR_POINTS = np.array([x for x, _ in POWER_CURVE], dtype=float)
P_POINTS = np.array([y for _, y in POWER_CURVE], dtype=float)

ALL_COLUMNS = [
    "time", "device", "active_power", "irradiance", "clear_sky_irradiance",
    "ambient_temp", "module_temp", "inverter_temp", "dc_voltage", "dc_current",
    "ac_voltage", "ac_current", "performance_ratio", "cloud_cover", "sun_elevation",
    "sun_azimuth", "day_length_hours", "fault_soiling", "fault_inverter_overheat",
    "fault_tracker_stuck", "fault_dc_string_outage", "is_faulted", "fault_labels",
    "downtime_start_time", "fault_severity",
]


In [4]:
def power_curve_w(irradiance):
    irradiance = np.asarray(irradiance, dtype=float)
    return np.interp(np.clip(irradiance, IRR_POINTS.min(), IRR_POINTS.max()), IRR_POINTS, P_POINTS)


def severity_label(values):
    values = np.asarray(values, dtype=float)
    labels = np.full(len(values), "none", dtype=object)
    labels[(values > 0.0) & (values < 0.33)] = "low"
    labels[(values >= 0.33) & (values < 0.66)] = "medium"
    labels[values >= 0.66] = "high"
    return labels


def build_fault_profile(n, start_idx, ramp_steps=1, end_idx=None, shape="linear", max_severity=1.0):
    profile = np.zeros(n, dtype=float)

    if start_idx is None or start_idx >= n:
        return profile

    end_idx = n if end_idx is None else min(end_idx, n)
    start_idx = max(0, start_idx)

    if end_idx <= start_idx:
        return profile

    if shape == "abrupt":
        profile[start_idx:end_idx] = max_severity
        return profile

    ramp_steps = max(1, int(ramp_steps))
    ramp_end = min(start_idx + ramp_steps, end_idx)
    profile[start_idx:ramp_end] = np.linspace(0.0, max_severity, ramp_end - start_idx, endpoint=False)
    profile[ramp_end:end_idx] = max_severity
    return profile


def add_outage_windows(n, outage_windows, steps_per_day):
    outage = np.zeros(n, dtype=float)
    for window in outage_windows:
        start_idx = window.get("start_idx")
        if start_idx is None and window.get("start_day") is not None:
            start_idx = int(window["start_day"] * steps_per_day)
        if start_idx is None:
            continue
        duration_steps = window.get("duration_steps")
        if duration_steps is None:
            duration_hours = window.get("duration_hours", 0.0)
            duration_steps = max(1, int(round(duration_hours * 6)))
        end_idx = min(n, start_idx + duration_steps)
        outage[start_idx:end_idx] = np.maximum(outage[start_idx:end_idx], window.get("severity", 1.0))
    return outage


def make_device_params(rng, overrides=None):
    params = {
        "dc_capacity_scale": float(np.clip(rng.normal(1.00, 0.02), 0.95, 1.05)),
        "voltage_bias": float(np.clip(rng.normal(0.0, 5.0), -15.0, 15.0)),
        "sensor_noise_scale": float(np.clip(rng.normal(1.0, 0.08), 0.85, 1.15)),
        "thermal_bias": float(np.clip(rng.normal(0.0, 0.8), -2.0, 2.0)),
        "dc_wiring_efficiency": float(np.clip(rng.normal(0.985, 0.005), 0.97, 0.995)),
        "tracker_gain": float(np.clip(rng.normal(1.0, 0.015), 0.96, 1.03)),
    }
    if overrides:
        params.update(overrides)
    return params


In [5]:
def solar_position(time_index, latitude_deg):
    latitude_rad = np.deg2rad(latitude_deg)
    day_of_year = time_index.dayofyear.to_numpy()
    hour = time_index.hour.to_numpy() + time_index.minute.to_numpy() / 60.0

    gamma = 2.0 * np.pi * (day_of_year - 1) / 365.0
    declination = (
        0.006918
        - 0.399912 * np.cos(gamma)
        + 0.070257 * np.sin(gamma)
        - 0.006758 * np.cos(2 * gamma)
        + 0.000907 * np.sin(2 * gamma)
        - 0.002697 * np.cos(3 * gamma)
        + 0.00148 * np.sin(3 * gamma)
    )

    equation_of_time = 229.18 * (
        0.000075
        + 0.001868 * np.cos(gamma)
        - 0.032077 * np.sin(gamma)
        - 0.014615 * np.cos(2 * gamma)
        - 0.040849 * np.sin(2 * gamma)
    )

    solar_time = hour + equation_of_time / 60.0
    hour_angle = np.deg2rad(15.0 * (solar_time - 12.0))

    sin_elevation = (
        np.sin(latitude_rad) * np.sin(declination)
        + np.cos(latitude_rad) * np.cos(declination) * np.cos(hour_angle)
    )
    sin_elevation = np.clip(sin_elevation, -1.0, 1.0)
    elevation_rad = np.arcsin(sin_elevation)

    azimuth_rad = np.arctan2(
        np.sin(hour_angle),
        np.cos(hour_angle) * np.sin(latitude_rad) - np.tan(declination) * np.cos(latitude_rad)
    )
    azimuth_deg = (np.rad2deg(azimuth_rad) + 180.0) % 360.0

    cos_h0 = -np.tan(latitude_rad) * np.tan(declination)
    cos_h0 = np.clip(cos_h0, -1.0, 1.0)
    h0_deg = np.rad2deg(np.arccos(cos_h0))
    day_length_hours = 2.0 * h0_deg / 15.0

    return np.rad2deg(elevation_rad), azimuth_deg, day_length_hours


def simulate_site_conditions(
    start="2025-01-01",
    n_days=365,
    freq="10min",
    latitude_deg=35.0,
    seed=random_seed,
):
    rng = np.random.default_rng(seed)
    steps_per_day = int(pd.Timedelta("1D") / pd.Timedelta(freq))
    time = pd.date_range(start=start, periods=n_days * steps_per_day, freq=freq)
    n = len(time)
    idx = np.arange(n)

    sun_elevation, sun_azimuth, day_length_hours = solar_position(time, latitude_deg)
    daylight_mask = sun_elevation > 0.0
    seasonal = np.sin(2.0 * np.pi * (time.dayofyear.to_numpy() / 365.25 - 0.28))
    diurnal = np.sin(2.0 * np.pi * (time.hour.to_numpy() + time.minute.to_numpy() / 60.0) / 24.0 - np.pi / 2)

    temp_noise = np.zeros(n)
    cloud_state = np.zeros(n)
    for i in range(1, n):
        temp_noise[i] = 0.985 * temp_noise[i - 1] + rng.normal(0.0, 0.18)
        cloud_state[i] = 0.94 * cloud_state[i - 1] + rng.normal(0.0, 0.20)

    ambient_temp = 18.0 + 11.0 * seasonal + 5.0 * diurnal + temp_noise

    summer_boost = np.clip((seasonal + 1.0) / 2.0, 0.0, 1.0)
    convective_clouds = np.clip((diurnal + 0.2) * summer_boost, 0.0, None)
    cloud_cover = 0.32 + 0.18 * np.sin(2.0 * np.pi * idx / (steps_per_day * 6.0) + 0.4) + 0.10 * convective_clouds + 0.20 * cloud_state
    cloud_cover = np.clip(cloud_cover, 0.02, 0.95)

    storm_centers = rng.integers(0, n, size=max(8, n_days // 30))
    storm_profile = np.zeros(n)
    for center in storm_centers:
        width = rng.integers(6, 36)
        depth = rng.uniform(0.10, 0.40)
        storm_profile += depth * np.exp(-0.5 * ((idx - center) / width) ** 2)
    cloud_cover = np.clip(cloud_cover + storm_profile, 0.02, 0.98)

    elevation_rad = np.deg2rad(np.clip(sun_elevation, 0.0, None))
    extraterrestrial = 1361.0 * (1.0 + 0.033 * np.cos(2.0 * np.pi * time.dayofyear.to_numpy() / 365.0))
    clear_sky_irradiance = extraterrestrial * np.power(np.sin(elevation_rad), 1.12)
    clear_sky_irradiance = np.clip(clear_sky_irradiance * 0.94, 0.0, 1250.0)

    transmittance = 1.0 - 0.72 * np.power(cloud_cover, 1.35)
    variability = 1.0 + rng.normal(0.0, 0.015, n)
    irradiance = clear_sky_irradiance * transmittance * variability
    irradiance = np.where(daylight_mask, irradiance, 0.0)
    irradiance = np.clip(irradiance, 0.0, clear_sky_irradiance)

    return pd.DataFrame({
        "time": time,
        "ambient_temp_site": ambient_temp,
        "cloud_cover_site": cloud_cover,
        "clear_sky_irradiance_site": clear_sky_irradiance,
        "irradiance_site": irradiance,
        "sun_elevation_site": np.clip(sun_elevation, -90.0, 90.0),
        "sun_azimuth_site": sun_azimuth,
        "day_length_hours_site": day_length_hours,
    })


In [6]:
def generate_healthy_device(site_df, device_id, device_params, seed=random_seed):
    rng = np.random.default_rng(seed)
    n = len(site_df)
    noise_scale = device_params["sensor_noise_scale"]

    ambient_temp = site_df["ambient_temp_site"].to_numpy() + rng.normal(0.0, 0.18 * noise_scale, n)
    cloud_cover = np.clip(site_df["cloud_cover_site"].to_numpy() + rng.normal(0.0, 0.015 * noise_scale, n), 0.0, 1.0)
    clear_sky_irradiance = np.clip(site_df["clear_sky_irradiance_site"].to_numpy() + rng.normal(0.0, 4.0 * noise_scale, n), 0.0, 1250.0)
    sun_elevation = site_df["sun_elevation_site"].to_numpy()
    sun_azimuth = site_df["sun_azimuth_site"].to_numpy()
    day_length_hours = site_df["day_length_hours_site"].to_numpy()

    tracker_efficiency = np.where(sun_elevation > 0.0, 0.985 + 0.010 * device_params["tracker_gain"], 0.0)
    thermal_derate = np.clip(1.0 - 0.0016 * np.clip(ambient_temp - 32.0, 0.0, None), 0.88, 1.0)
    dc_efficiency = device_params["dc_wiring_efficiency"]

    irradiance = site_df["irradiance_site"].to_numpy() * tracker_efficiency
    irradiance = irradiance * device_params["dc_capacity_scale"] * thermal_derate * dc_efficiency
    irradiance = np.where(sun_elevation > 0.0, irradiance, 0.0)
    irradiance = np.clip(irradiance + rng.normal(0.0, 5.0 * noise_scale, n), 0.0, 1250.0)

    active_power = power_curve_w(irradiance)
    active_power = np.where(sun_elevation > 0.0, active_power, 0.0)

    module_temp = ambient_temp + 0.028 * irradiance + 0.8 * cloud_cover + device_params["thermal_bias"]
    inverter_temp = ambient_temp + 0.0105 * active_power + 0.35 * cloud_cover + 0.5 * device_params["thermal_bias"]

    dc_voltage = 820.0 - 0.55 * np.clip(module_temp - 25.0, -25.0, 55.0) + device_params["voltage_bias"]
    dc_voltage = np.where(active_power > 1.0, dc_voltage, 0.0)
    dc_voltage = np.clip(dc_voltage, 0.0, 950.0)

    dc_current = np.divide(active_power * 1.03, np.maximum(dc_voltage, 1.0), out=np.zeros(n), where=dc_voltage > 0.0)
    ac_voltage = np.where(active_power > 1.0, 400.0 + rng.normal(0.0, 2.0 * noise_scale, n), 0.0)
    ac_current = np.divide(active_power, np.sqrt(3.0) * np.maximum(ac_voltage, 1.0), out=np.zeros(n), where=ac_voltage > 0.0)

    clear_power = power_curve_w(np.clip(clear_sky_irradiance, 0.0, 1250.0))
    performance_ratio = np.divide(active_power, np.maximum(clear_power, 1.0), out=np.zeros(n), where=clear_power > 1.0)
    performance_ratio = np.clip(performance_ratio, 0.0, 1.02)

    df = pd.DataFrame({
        "time": site_df["time"].to_numpy(),
        "device": device_id,
        "active_power": active_power,
        "irradiance": irradiance,
        "clear_sky_irradiance": clear_sky_irradiance,
        "ambient_temp": ambient_temp,
        "module_temp": module_temp,
        "inverter_temp": inverter_temp,
        "dc_voltage": dc_voltage,
        "dc_current": dc_current,
        "ac_voltage": ac_voltage,
        "ac_current": ac_current,
        "performance_ratio": performance_ratio,
        "cloud_cover": cloud_cover,
        "sun_elevation": sun_elevation,
        "sun_azimuth": sun_azimuth,
        "day_length_hours": day_length_hours,
    })

    for col in ["fault_soiling", "fault_inverter_overheat", "fault_tracker_stuck", "fault_dc_string_outage"]:
        df[col] = 0

    return df


In [7]:
def apply_soiling_fault(df, fault_config):
    profile = build_fault_profile(
        n=len(df),
        start_idx=fault_config["start_idx"],
        ramp_steps=fault_config.get("ramp_steps", 1),
        end_idx=fault_config.get("end_idx"),
        shape=fault_config.get("shape", "linear"),
        max_severity=fault_config.get("max_severity", 1.0),
    )

    if np.all(profile == 0):
        return df, profile

    sunlight_factor = np.clip(df["irradiance"].to_numpy() / 900.0, 0.0, 1.0)
    transmittance_loss = 0.03 + 0.14 * profile
    power_factor = 1.0 - transmittance_loss * sunlight_factor

    df["irradiance"] *= power_factor
    df["active_power"] = power_curve_w(df["irradiance"].to_numpy())
    df["module_temp"] += 1.2 * profile * sunlight_factor
    df["performance_ratio"] *= np.clip(power_factor, 0.0, 1.0)

    return df, profile


def apply_inverter_overheat_fault(df, fault_config):
    n = len(df)
    profile = np.zeros(n, dtype=float)

    start_idx = fault_config.get("start_idx", 0)
    end_idx = fault_config.get("end_idx", n)
    temp_threshold = fault_config.get("temp_threshold", 34.0)
    irradiance_threshold = fault_config.get("irradiance_threshold", 780.0)
    midday_start = fault_config.get("midday_start", 11.0)
    midday_end = fault_config.get("midday_end", 16.5)

    hour = df["time"].dt.hour.to_numpy() + df["time"].dt.minute.to_numpy() / 60.0
    hot_midday = (
        (np.arange(n) >= start_idx)
        & (np.arange(n) < end_idx)
        & (df["ambient_temp"].to_numpy() >= temp_threshold)
        & (df["irradiance"].to_numpy() >= irradiance_threshold)
        & (hour >= midday_start)
        & (hour <= midday_end)
    )

    if not np.any(hot_midday):
        return df, profile

    base = np.clip((df["ambient_temp"].to_numpy() - temp_threshold) / 10.0, 0.0, 1.0)
    base += np.clip((df["irradiance"].to_numpy() - irradiance_threshold) / 350.0, 0.0, 1.0)
    intermittent = (np.sin(np.arange(n) / 2.3) > fault_config.get("trip_trigger", 0.15)).astype(float)
    profile = np.where(hot_midday, np.clip(0.45 * base + 0.55 * intermittent, 0.0, 1.0), 0.0)

    thermal_boost = 10.0 * np.clip(profile, 0.0, 1.0)
    trip_mask = profile >= fault_config.get("trip_level", 0.70)
    derate_mask = (profile > 0.0) & (~trip_mask)

    df.loc[derate_mask, "active_power"] *= (1.0 - 0.22 * profile[derate_mask])
    df.loc[trip_mask, ["active_power", "dc_current", "ac_current"]] = 0.0
    df.loc[trip_mask, "ac_voltage"] = 0.0
    df.loc[trip_mask, "performance_ratio"] = 0.0
    df["inverter_temp"] += thermal_boost

    return df, profile


def apply_tracker_stuck_fault(df, fault_config):
    profile = build_fault_profile(
        n=len(df),
        start_idx=fault_config["start_idx"],
        ramp_steps=fault_config.get("ramp_steps", 1),
        end_idx=fault_config.get("end_idx"),
        shape=fault_config.get("shape", "linear"),
        max_severity=fault_config.get("max_severity", 1.0),
    )

    if np.all(profile == 0):
        return df, profile

    hour = df["time"].dt.hour.to_numpy() + df["time"].dt.minute.to_numpy() / 60.0
    shoulder_factor = np.clip(np.abs(hour - 12.0) / 5.0, 0.0, 1.0)
    sun_factor = np.clip(df["sun_elevation"].to_numpy() / 45.0, 0.0, 1.0)
    tracking_loss = 0.05 + 0.32 * profile * shoulder_factor * (1.15 - 0.35 * sun_factor)
    power_factor = np.clip(1.0 - tracking_loss, 0.55, 1.0)

    df["irradiance"] *= power_factor
    df["active_power"] = power_curve_w(df["irradiance"].to_numpy())
    df["module_temp"] += 0.6 * profile * shoulder_factor
    df["performance_ratio"] *= power_factor

    return df, profile


def apply_dc_string_outage_fault(df, fault_config):
    profile = build_fault_profile(
        n=len(df),
        start_idx=fault_config["start_idx"],
        ramp_steps=fault_config.get("ramp_steps", 1),
        end_idx=fault_config.get("end_idx"),
        shape=fault_config.get("shape", "abrupt"),
        max_severity=fault_config.get("max_severity", 1.0),
    )

    if np.all(profile == 0):
        return df, profile

    current_loss = 0.10 + 0.22 * profile
    power_factor = 1.0 - current_loss

    df["active_power"] *= power_factor
    df["dc_current"] *= power_factor
    df["dc_voltage"] *= 1.0 - 0.03 * profile
    df["performance_ratio"] *= power_factor
    df["inverter_temp"] += 1.0 * profile

    return df, profile


In [8]:
FAULT_REGISTRY = {
    "soiling": apply_soiling_fault,
    "inverter_overheat": apply_inverter_overheat_fault,
    "tracker_stuck": apply_tracker_stuck_fault,
    "dc_string_outage": apply_dc_string_outage_fault,
}

FAULT_COLUMN_MAP = {
    "soiling": "fault_soiling",
    "inverter_overheat": "fault_inverter_overheat",
    "tracker_stuck": "fault_tracker_stuck",
    "dc_string_outage": "fault_dc_string_outage",
}


def apply_fault(df, fault_config):
    fault_type = fault_config["type"]
    if fault_type not in FAULT_REGISTRY:
        raise ValueError(f"Unknown fault type: {fault_type}")
    return FAULT_REGISTRY[fault_type](df, fault_config)


def finalize_fault_labels(df, fault_states, downtime_profile, fault_start_times):
    n = len(df)

    if not fault_states and np.all(downtime_profile == 0):
        df["is_faulted"] = 0
        df["fault_labels"] = "healthy"
        df["downtime_start_time"] = pd.NaT
        df["fault_severity"] = "none"
        return df

    profiles = list(fault_states.values()) if fault_states else [np.zeros(n)]
    max_profile = np.maximum.reduce(profiles + [downtime_profile])
    df["is_faulted"] = (max_profile > 0).astype(int)
    df["fault_severity"] = severity_label(max_profile)

    labels = np.full(n, "healthy", dtype=object)
    for fault_type, profile in fault_states.items():
        active = profile > 0
        df[FAULT_COLUMN_MAP[fault_type]] = active.astype(int)
        labels = np.where(active & (labels == "healthy"), fault_type, labels)
        labels = np.where(active & (labels != "healthy") & (labels != fault_type), labels + "|" + fault_type, labels)

    downtime_active = downtime_profile > 0
    labels = np.where(downtime_active & (labels == "healthy"), "downtime", labels)
    labels = np.where(downtime_active & (labels != "healthy") & (labels != "downtime"), labels + "|downtime", labels)

    downtime_start_time = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")
    if fault_start_times:
        downtime_start_time[:] = min(fault_start_times)

    df["fault_labels"] = labels
    df["downtime_start_time"] = downtime_start_time
    return df


def apply_outages(df, outage_profile):
    if np.all(outage_profile == 0):
        return df

    outage_mask = outage_profile > 0
    df.loc[outage_mask, ["active_power", "dc_current", "ac_current"]] = 0.0
    df.loc[outage_mask, "ac_voltage"] = 0.0
    df.loc[outage_mask, "performance_ratio"] = 0.0
    return df


In [9]:
def round_and_clip(df):
    night_mask = (df["sun_elevation"] <= 0.5) | (df["irradiance"] <= 0.5)
    df.loc[night_mask, ["active_power", "irradiance", "dc_voltage", "dc_current", "ac_voltage", "ac_current", "performance_ratio"]] = 0.0

    df["active_power"] = np.round(np.clip(df["active_power"], 0.0, 4400.0), 2)
    df["irradiance"] = np.round(np.clip(df["irradiance"], 0.0, 1250.0), 2)
    df["clear_sky_irradiance"] = np.round(np.clip(df["clear_sky_irradiance"], 0.0, 1250.0), 2)
    df["ambient_temp"] = np.round(np.clip(df["ambient_temp"], -20.0, 55.0), 2)
    df["module_temp"] = np.round(np.clip(df["module_temp"], -20.0, 95.0), 2)
    df["inverter_temp"] = np.round(np.clip(df["inverter_temp"], -20.0, 100.0), 2)
    df["dc_voltage"] = np.round(np.clip(df["dc_voltage"], 0.0, 950.0), 2)
    df["dc_current"] = np.round(np.clip(df["dc_current"], 0.0, 12.0), 3)
    df["ac_voltage"] = np.round(np.clip(df["ac_voltage"], 0.0, 460.0), 2)
    df["ac_current"] = np.round(np.clip(df["ac_current"], 0.0, 8.0), 3)
    df["performance_ratio"] = np.round(np.clip(df["performance_ratio"], 0.0, 1.05), 4)
    df["cloud_cover"] = np.round(np.clip(df["cloud_cover"], 0.0, 1.0), 3)
    df["sun_elevation"] = np.round(np.clip(df["sun_elevation"], -90.0, 90.0), 2)
    df["sun_azimuth"] = np.round(df["sun_azimuth"] % 360.0, 2)
    df["day_length_hours"] = np.round(np.clip(df["day_length_hours"], 0.0, 24.0), 2)
    return df


In [10]:
def simulate_device(site_df, device_config, seed=random_seed):
    rng = np.random.default_rng(seed)
    steps_per_day = int(pd.Timedelta("1D") / pd.Timedelta(device_config.get("freq", "10min")))
    device_params = make_device_params(rng, device_config.get("device_params"))
    df = generate_healthy_device(site_df, device_config["device_id"], device_params, seed=seed)

    fault_states = {}
    fault_start_times = []

    for fault in device_config.get("faults", []):
        fault_cfg = dict(fault)
        if "start_idx" not in fault_cfg and fault_cfg.get("start_day") is not None:
            fault_cfg["start_idx"] = int(fault_cfg["start_day"] * steps_per_day)
        if "end_idx" not in fault_cfg and fault_cfg.get("end_day") is not None:
            fault_cfg["end_idx"] = int(fault_cfg["end_day"] * steps_per_day)
        if "ramp_steps" not in fault_cfg:
            fault_cfg["ramp_steps"] = max(1, int(fault_cfg.get("ramp_days", 1) * steps_per_day))

        df, profile = apply_fault(df, fault_cfg)
        fault_states[fault_cfg["type"]] = np.maximum(fault_states.get(fault_cfg["type"], np.zeros(len(df))), profile)

        if np.any(profile > 0):
            first_idx = int(np.where(profile > 0)[0][0])
            fault_start_times.append(site_df.iloc[first_idx]["time"])

    outage_profile = add_outage_windows(len(df), device_config.get("outages", []), steps_per_day)
    if np.any(outage_profile > 0):
        outage_indices = np.where(outage_profile > 0)[0]
        fault_start_times.append(site_df.iloc[int(outage_indices[0])]["time"])
        df = apply_outages(df, outage_profile)

    df = finalize_fault_labels(df, fault_states, outage_profile, fault_start_times)
    df = round_and_clip(df)
    return df[ALL_COLUMNS]


def build_fault_event_table(site_df, device_configs, freq="10min"):
    steps_per_day = int(pd.Timedelta("1D") / pd.Timedelta(freq))
    rows = []

    for cfg in device_configs:
        for fault in cfg.get("faults", []):
            start_idx = fault.get("start_idx")
            if start_idx is None and fault.get("start_day") is not None:
                start_idx = int(fault["start_day"] * steps_per_day)
            end_idx = fault.get("end_idx")
            if end_idx is None and fault.get("end_day") is not None:
                end_idx = int(fault["end_day"] * steps_per_day)

            rows.append({
                "device": cfg["device_id"],
                "event_type": fault["type"],
                "start_time": site_df.iloc[start_idx]["time"] if start_idx is not None and start_idx < len(site_df) else pd.NaT,
                "end_time": site_df.iloc[end_idx]["time"] if end_idx is not None and end_idx < len(site_df) else pd.NaT,
                "shape": fault.get("shape", "linear"),
                "max_severity": fault.get("max_severity", 1.0),
            })

        for outage in cfg.get("outages", []):
            start_idx = outage.get("start_idx")
            if start_idx is None and outage.get("start_day") is not None:
                start_idx = int(outage["start_day"] * steps_per_day)
            duration_steps = outage.get("duration_steps")
            if duration_steps is None:
                duration_steps = max(1, int(round(outage.get("duration_hours", 0.0) * 6)))
            end_idx = start_idx + duration_steps if start_idx is not None else None

            rows.append({
                "device": cfg["device_id"],
                "event_type": "downtime",
                "start_time": site_df.iloc[start_idx]["time"] if start_idx is not None and start_idx < len(site_df) else pd.NaT,
                "end_time": site_df.iloc[end_idx]["time"] if end_idx is not None and end_idx < len(site_df) else pd.NaT,
                "shape": "abrupt",
                "max_severity": outage.get("severity", 1.0),
            })

    return pd.DataFrame(rows)


def simulate_farm(site_config, device_configs, seed=random_seed):
    site_df = simulate_site_conditions(**site_config, seed=seed)
    all_devices = []
    for i, cfg in enumerate(device_configs):
        cfg = dict(cfg)
        cfg["freq"] = site_config.get("freq", "10min")
        all_devices.append(simulate_device(site_df, cfg, seed=seed + i + 1))

    data = pd.concat(all_devices, ignore_index=True).sort_values(["time", "device"]).reset_index(drop=True)
    events = build_fault_event_table(site_df, device_configs, freq=site_config.get("freq", "10min"))
    return data, events


#### Solar data usually saved at a 5min level, so we will do that.

In [11]:
site_config = {
    "start": "2025-01-01",
    "n_days": 365,
    "freq": "5min",
    "latitude_deg": 35.0,
}

device_configs = [
    {
        "device_id": "PV000",
        "device_params": {"dc_capacity_scale": 1.00},
        "faults": [],
        "outages": [],
    },
    {
        "device_id": "PV001",
        "device_params": {"dc_capacity_scale": 0.995},
        "faults": [
            {
                "type": "soiling",
                "start_day": 120,
                "end_day": 280,
                "ramp_days": 35,
                "shape": "linear",
                "max_severity": 1.0,
            }
        ],
        "outages": [],
    },
    {
        "device_id": "PV002",
        "device_params": {"dc_capacity_scale": 1.005, "thermal_bias": 1.2},
        "faults": [
            {
                "type": "inverter_overheat",
                "start_day": 170,
                "end_day": 255,
                "temp_threshold": 34.0,
                "irradiance_threshold": 780.0,
                "trip_level": 0.70,
            }
        ],
        "outages": [
            {"start_day": 210.0, "duration_hours": 3.0, "severity": 1.0},
            {"start_day": 228.25, "duration_hours": 2.5, "severity": 1.0},
        ],
    },
    {
        "device_id": "PV003",
        "device_params": {"tracker_gain": 0.99},
        "faults": [
            {
                "type": "tracker_stuck",
                "start_day": 85,
                "end_day": 225,
                "ramp_days": 2,
                "shape": "abrupt",
                "max_severity": 1.0,
            }
        ],
        "outages": [
            {"start_day": 226.0, "duration_hours": 28.0, "severity": 1.0},
        ],
    },
    {
        "device_id": "PV004",
        "device_params": {"dc_capacity_scale": 0.99, "voltage_bias": -8.0},
        "faults": [
            {
                "type": "dc_string_outage",
                "start_day": 295,
                "ramp_days": 1,
                "shape": "abrupt",
                "max_severity": 1.0,
            }
        ],
        "outages": [
            {"start_day": 320.5, "duration_hours": 5.0, "severity": 1.0},
            {"start_day": 333.0, "duration_hours": 4.0, "severity": 1.0},
        ],
    },
    {
        "device_id": "PV005",
        "device_params": {"dc_capacity_scale": 0.998},
        "faults": [
            {
                "type": "soiling",
                "start_day": 30,
                "end_day": 110,
                "ramp_days": 25,
                "shape": "linear",
                "max_severity": 1.0,
            }
        ],
        "outages": [],
    },
    {
        "device_id": "PV006",
        "device_params": {"tracker_gain": 0.985},
        "faults": [
            {
                "type": "tracker_stuck",
                "start_day": 145,
                "end_day": 200,
                "ramp_days": 1,
                "shape": "abrupt",
                "max_severity": 1.0,
            }
        ],
        "outages": [
            {"start_day": 200.5, "duration_hours": 6.0, "severity": 1.0},
        ],
    },
    {
        "device_id": "PV007",
        "device_params": {"dc_capacity_scale": 0.99, "voltage_bias": -5.0},
        "faults": [
            {
                "type": "dc_string_outage",
                "start_day": 60,
                "end_day": 115,
                "ramp_days": 1,
                "shape": "abrupt",
                "max_severity": 1.0,
            }
        ],
        "outages": [
            {"start_day": 115.0, "duration_hours": 4.0, "severity": 1.0},
        ],
    },
    {
        "device_id": "PV008",
        "device_params": {"dc_capacity_scale": 1.003, "thermal_bias": 1.0},
        "faults": [
            {
                "type": "inverter_overheat",
                "start_day": 195,
                "end_day": 245,
                "temp_threshold": 34.0,
                "irradiance_threshold": 780.0,
                "trip_level": 0.70,
            }
        ],
        "outages": [
            {"start_day": 245.25, "duration_hours": 3.5, "severity": 1.0},
        ],
    },
    {
        "device_id": "PV009",
        "device_params": {"dc_capacity_scale": 0.997},
        "faults": [
            {
                "type": "soiling",
                "start_day": 240,
                "end_day": 340,
                "ramp_days": 30,
                "shape": "linear",
                "max_severity": 1.0,
            }
        ],
        "outages": [],
    },
    {
        "device_id": "PV010",
        "device_params": {"dc_capacity_scale": 0.995, "tracker_gain": 0.99},
        "faults": [
            {
                "type": "soiling",
                "start_day": 40,
                "end_day": 130,
                "ramp_days": 25,
                "shape": "linear",
                "max_severity": 1.0,
            },
            {
                "type": "tracker_stuck",
                "start_day": 200,
                "end_day": 290,
                "ramp_days": 1,
                "shape": "abrupt",
                "max_severity": 1.0,
            },
        ],
        "outages": [
            {"start_day": 290.25, "duration_hours": 5.0, "severity": 1.0},
        ],
    },
]


In [12]:
df, events = simulate_farm(site_config, device_configs, seed=random_seed)

output_dir = Path("../generated_data/time_series/solar_data_with_repair")
output_dir.mkdir(parents=True, exist_ok=True)

df.to_parquet(output_dir / "solar_farm_stream_data_with_repair.parquet", index=False)
events.to_parquet(output_dir / "solar_farm_fault_events_with_repair.parquet", index=False)

In [13]:
df.groupby("device")[["active_power", "is_faulted"]].mean().round(3)

,active_power,is_faulted
device,,
PV000,999.171,0.000
PV001,939.781,0.438
PV002,1004.772,0.005
PV003,878.719,0.385
PV004,866.535,0.192
PV005,970.155,0.219
PV006,942.458,0.151
PV007,869.548,0.151
PV008,1005.280,0.003


In [14]:
df["fault_labels"].value_counts().head(20)

fault_labels
healthy                      913288
soiling                      123836
tracker_stuck                 82080
dc_string_outage              35946
inverter_overheat               804
downtime                        312
dc_string_outage|downtime        54
Name: count, dtype: int64

In [15]:
events

,device,event_type,start_time,end_time,shape,max_severity
0,PV001,soiling,2025-05-01 00:00:00,2025-10-08 00:00:00,linear,1.0
1,PV002,inverter_overheat,2025-06-20 00:00:00,2025-09-13 00:00:00,linear,1.0
2,PV002,downtime,2025-07-30 00:00:00,2025-07-30 01:30:00,abrupt,1.0
3,PV002,downtime,2025-08-17 06:00:00,2025-08-17 07:15:00,abrupt,1.0
4,PV003,tracker_stuck,2025-03-27 00:00:00,2025-08-14 00:00:00,abrupt,1.0
5,PV003,downtime,2025-08-15 00:00:00,2025-08-15 14:00:00,abrupt,1.0
6,PV004,dc_string_outage,2025-10-23 00:00:00,NaT,abrupt,1.0
7,PV004,downtime,2025-11-17 12:00:00,2025-11-17 14:30:00,abrupt,1.0
8,PV004,downtime,2025-11-30 00:00:00,2025-11-30 02:00:00,abrupt,1.0
9,PV005,soiling,2025-01-31 00:00:00,2025-04-21 00:00:00,linear,1.0


In [16]:
df

,time,device,active_power,irradiance,clear_sky_irradiance,ambient_temp,module_temp,inverter_temp,dc_voltage,dc_current,...,sun_azimuth,day_length_hours,fault_soiling,fault_inverter_overheat,fault_tracker_stuck,fault_dc_string_outage,is_faulted,fault_labels,downtime_start_time,fault_severity
0,2025-01-01 00:00:00,PV000,0.0,0.0,4.88,2.20,1.90,1.99,0.0,0.0,...,356.77,9.69,0,0,0,0,0,healthy,NaT,none
1,2025-01-01 00:00:00,PV001,0.0,0.0,1.29,2.43,3.65,3.02,0.0,0.0,...,356.77,9.69,0,0,0,0,0,healthy,2025-05-01 00:05:00,none
2,2025-01-01 00:00:00,PV002,0.0,0.0,1.08,2.08,3.60,2.81,0.0,0.0,...,356.77,9.69,0,0,0,0,0,healthy,2025-06-21 13:30:00,none
3,2025-01-01 00:00:00,PV003,0.0,0.0,0.00,2.07,2.97,2.50,0.0,0.0,...,356.77,9.69,0,0,0,0,0,healthy,2025-03-27 00:00:00,none
4,2025-01-01 00:00:00,PV004,0.0,0.0,0.15,2.06,0.81,1.41,0.0,0.0,...,356.77,9.69,0,0,0,0,0,healthy,2025-10-23 00:00:00,none
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1156315,2025-12-31 23:55:00,PV006,0.0,0.0,3.94,0.68,-0.39,0.14,0.0,0.0,...,351.72,9.68,0,0,0,0,0,healthy,2025-05-26 00:00:00,none
1156316,2025-12-31 23:55:00,PV007,0.0,0.0,7.94,0.30,-0.66,-0.19,0.0,0.0,...,351.72,9.68,0,0,0,0,0,healthy,2025-03-02 00:00:00,none
1156317,2025-12-31 23:55:00,PV008,0.0,0.0,0.00,0.48,1.61,1.04,0.0,0.0,...,351.72,9.68,0,0,0,0,0,healthy,2025-07-15 11:00:00,none
1156318,2025-12-31 23:55:00,PV009,0.0,0.0,0.00,0.42,0.53,0.44,0.0,0.0,...,351.72,9.68,0,0,0,0,0,healthy,2025-08-29 00:05:00,none


In [17]:
df["fault_severity"].value_counts()

fault_severity
none      913288
high      220685
low        11263
medium     11084
Name: count, dtype: int64

In [18]:
events

,device,event_type,start_time,end_time,shape,max_severity
0,PV001,soiling,2025-05-01 00:00:00,2025-10-08 00:00:00,linear,1.0
1,PV002,inverter_overheat,2025-06-20 00:00:00,2025-09-13 00:00:00,linear,1.0
2,PV002,downtime,2025-07-30 00:00:00,2025-07-30 01:30:00,abrupt,1.0
3,PV002,downtime,2025-08-17 06:00:00,2025-08-17 07:15:00,abrupt,1.0
4,PV003,tracker_stuck,2025-03-27 00:00:00,2025-08-14 00:00:00,abrupt,1.0
5,PV003,downtime,2025-08-15 00:00:00,2025-08-15 14:00:00,abrupt,1.0
6,PV004,dc_string_outage,2025-10-23 00:00:00,NaT,abrupt,1.0
7,PV004,downtime,2025-11-17 12:00:00,2025-11-17 14:30:00,abrupt,1.0
8,PV004,downtime,2025-11-30 00:00:00,2025-11-30 02:00:00,abrupt,1.0
9,PV005,soiling,2025-01-31 00:00:00,2025-04-21 00:00:00,linear,1.0
